# HistGBM hyperparameter tuning

In [1]:
# Load data

import pandas as pd
import numpy as np 

train_df = pd.read_parquet("data/train_data.parquet")

train_df.index = pd.to_numeric(train_df.index, errors="coerce")

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_df = train_df.drop(columns=cols_to_drop, errors="ignore")

In [2]:
# Separate features and targets

train_full_X = train_df.drop(["target", "target_annual_roi"], axis=1)
train_full_y_cat = train_df["target"]
train_full_y_reg = train_df["target_annual_roi"]

# Drop datetime features from the feature set

cols_to_drop = ["issue_d", "earliest_cr_line"]
train_full_X = train_full_X.drop(columns=cols_to_drop, errors="ignore")

In [3]:
# Create subsets of the data for different training sizes - chronplogical order is preserved, so we take the last N rows for each subset

train_1k_X = train_full_X.tail(1000)
train_1k_y_cat = train_full_y_cat.tail(1000)
train_1k_y_reg = train_full_y_reg.tail(1000)

train_10k_X = train_full_X.tail(10000)
train_10k_y_cat = train_full_y_cat.tail(10000)
train_10k_y_reg = train_full_y_reg.tail(10000)

train_100k_X = train_full_X.tail(100000)
train_100k_y_cat = train_full_y_cat.tail(100000)
train_100k_y_reg = train_full_y_reg.tail(100000)

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

cols_to_nominal_cat = train_df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical columns:")
for col in cols_to_nominal_cat:
    print(f"- {col}")

cardinality = train_df[cols_to_nominal_cat].nunique()
threshold_for_ohe = 5

cols_for_ohe = cardinality[cardinality <= threshold_for_ohe].index.tolist()
cols_for_te = cardinality[cardinality > threshold_for_ohe].index.tolist()

ohe_categories = []
for col in cols_for_ohe:
    unique_cats = train_df[col].dropna().unique().tolist()
    ohe_categories.append(unique_cats)

ohe_transformer = OneHotEncoder(
    categories=ohe_categories, 
    drop="if_binary", 
    handle_unknown="ignore", 
    sparse_output=False
)

target_transformer_nominal = TargetEncoder(target_type="binary", smooth="auto")

numeric_preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", ohe_transformer, cols_for_ohe),
        ("target_enc", target_transformer_nominal, cols_for_te)
    ],
    remainder="passthrough", 
    verbose_feature_names_out=False
).set_output(transform="pandas")

imputed_numeric_preprocessor = make_pipeline(
    numeric_preprocessor,
    SimpleImputer(strategy="median")
).set_output(transform="pandas")

gbdt_preprocessor = "passthrough"

Categorical columns identified for encoding:
- home_ownership
- verification_status
- purpose
- addr_state
- initial_list_status
- application_type
- disbursement_method


## Classification

[Parameters](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.GradientBoostingRegressor.html)

### 1k

In [5]:
# Check the dates of 1k subset to ensure all data is from same month

print(train_1k_X["issue_d_month"].unique())
print(train_1k_X["issue_d_year"].unique())

<IntegerArray>
[10]
Length: 1, dtype: Int64
<IntegerArray>
[2016]
Length: 1, dtype: Int64


In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline


# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_cat.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_cat.head(200)

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
        "class_weight": "balanced", # Adjust weights inversely proportional to class frequencies
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_1k_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_1k_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_1k_parallel.html")


[I 2026-04-23 18:06:55,154] A new study created in memory with name: no-name-e2d1fde6-6bbe-480c-88c3-45a687179b61
[I 2026-04-23 18:07:03,701] Trial 0 finished with value: 0.6362156043190527 and parameters: {'max_iter': 134, 'learning_rate': 0.13530940674051317, 'max_depth': 8, 'min_samples_leaf': 33, 'max_features': 0.5784642793890922, 'max_leaf_nodes': 16, 'l2_regularization': 0.003650169766978062, 'max_bins': 118}. Best is trial 0 with value: 0.6362156043190527.
[I 2026-04-23 18:07:08,792] Trial 6 finished with value: 0.6573157493847149 and parameters: {'max_iter': 401, 'learning_rate': 0.036516485573623206, 'max_depth': 6, 'min_samples_leaf': 62, 'max_features': 0.7385883856102459, 'max_leaf_nodes': 12, 'l2_regularization': 25.686238771563822, 'max_bins': 99}. Best is trial 6 with value: 0.6573157493847149.
[I 2026-04-23 18:07:09,087] Trial 1 finished with value: 0.6588034283723939 and parameters: {'max_iter': 570, 'learning_rate': 0.02531000436046725, 'max_depth': 7, 'min_samples_l


BEST AUC: 0.6721
BEST PARAMETERS:
best_params = {
    "max_iter": 516,
    "learning_rate": 0.006874060229610276,
    "max_depth": 4,
    "min_samples_leaf": 67,
    "max_features": 0.20074088835295997,
    "max_leaf_nodes": 28,
    "l2_regularization": 2.0167880959766253,
    "max_bins": 75,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.4322
  min_samples_leaf    : 0.2609
  max_iter            : 0.1397
  max_depth           : 0.0968
  max_bins            : 0.0360
  max_leaf_nodes      : 0.0168
  max_features        : 0.0127
  l2_regularization   : 0.0050


In [ ]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_histgbm.best_params.copy()
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")


final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingClassifier(**best_params)
)

final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print(f"Optuna Val AUC: {study_histgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")

Optuna Val AUC: 0.6721
Holdout Test AUC: 0.6711


### 10k

In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline


# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_cat.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_cat.tail(2000)

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
        "class_weight": "balanced", # Adjust weights inversely proportional to class frequencies
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_10k_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_10k_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_10k_parallel.html")

[I 2026-04-23 18:07:58,167] A new study created in memory with name: no-name-98051177-ba99-4276-a921-5ff8debf1359
[I 2026-04-23 18:08:21,114] Trial 6 finished with value: 0.6576115149350054 and parameters: {'max_iter': 530, 'learning_rate': 0.18185051856144688, 'max_depth': 2, 'min_samples_leaf': 36, 'max_features': 0.5113786798055664, 'max_leaf_nodes': 16, 'l2_regularization': 1.4033742160815596e-05, 'max_bins': 130}. Best is trial 6 with value: 0.6576115149350054.
[I 2026-04-23 18:08:29,312] Trial 1 finished with value: 0.6722964909824029 and parameters: {'max_iter': 729, 'learning_rate': 0.06594203537073429, 'max_depth': 2, 'min_samples_leaf': 62, 'max_features': 0.7532022560904119, 'max_leaf_nodes': 31, 'l2_regularization': 1.1497390402100018, 'max_bins': 214}. Best is trial 1 with value: 0.6722964909824029.
[I 2026-04-23 18:08:33,271] Trial 4 finished with value: 0.6887849750541467 and parameters: {'max_iter': 891, 'learning_rate': 0.01231556730989468, 'max_depth': 7, 'min_samples


BEST AUC: 0.6888
BEST PARAMETERS:
best_params = {
    "max_iter": 891,
    "learning_rate": 0.01231556730989468,
    "max_depth": 7,
    "min_samples_leaf": 38,
    "max_features": 0.6770980238359037,
    "max_leaf_nodes": 3,
    "l2_regularization": 0.00023905859551234077,
    "max_bins": 189,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.4200
  max_iter            : 0.3379
  max_depth           : 0.1045
  min_samples_leaf    : 0.0419
  max_bins            : 0.0405
  max_features        : 0.0347
  max_leaf_nodes      : 0.0203
  l2_regularization   : 0.0003


In [ ]:
best_params = study_histgbm.best_params.copy()

best_params["max_iter"] = int(best_params["max_iter"] * 2)
best_params["learning_rate"] = best_params["learning_rate"] / 2

best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingClassifier(**best_params)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_histgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)


Optuna Val AUC:   0.6888
Holdout Test AUC: 0.6930


### 100k

In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline


# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_cat.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_cat.tail(20000)

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
        "class_weight": "balanced", # Adjust weights inversely proportional to class frequencies
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_100k_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_100k_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_100k_parallel.html")

[I 2026-04-23 18:09:40,379] A new study created in memory with name: no-name-5431847f-14d9-48fd-b510-0cb7eee7c587
[I 2026-04-23 18:10:03,123] Trial 7 finished with value: 0.7082125544668492 and parameters: {'max_iter': 791, 'learning_rate': 0.09096169288442667, 'max_depth': 2, 'min_samples_leaf': 227, 'max_features': 0.269539192840668, 'max_leaf_nodes': 105, 'l2_regularization': 4.534818340752345, 'max_bins': 163}. Best is trial 7 with value: 0.7082125544668492.
[I 2026-04-23 18:11:01,741] Trial 2 finished with value: 0.6962132602005531 and parameters: {'max_iter': 488, 'learning_rate': 0.1494222725487687, 'max_depth': 15, 'min_samples_leaf': 96, 'max_features': 0.6783351417801418, 'max_leaf_nodes': 143, 'l2_regularization': 0.014159334522610392, 'max_bins': 101}. Best is trial 7 with value: 0.7082125544668492.
[I 2026-04-23 18:13:42,934] Trial 5 finished with value: 0.7090307127711869 and parameters: {'max_iter': 578, 'learning_rate': 0.021990644865682203, 'max_depth': 11, 'min_sample


BEST AUC: 0.7093
BEST PARAMETERS:
best_params = {
    "max_iter": 726,
    "learning_rate": 0.008840777878260282,
    "max_depth": 16,
    "min_samples_leaf": 391,
    "max_features": 0.4278099693772527,
    "max_leaf_nodes": 158,
    "l2_regularization": 1.7686171043220593,
    "max_bins": 207,
}

--- PARAMETER IMPORTANCE ---
  max_iter            : 0.5445
  min_samples_leaf    : 0.1287
  max_depth           : 0.1113
  l2_regularization   : 0.0660
  max_features        : 0.0623
  max_leaf_nodes      : 0.0514
  learning_rate       : 0.0303
  max_bins            : 0.0055


In [ ]:
best_params = study_histgbm.best_params.copy()

best_params["max_iter"] = int(best_params["max_iter"] * 10)
best_params["learning_rate"] = best_params["learning_rate"] / 10

best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingClassifier(**best_params)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_histgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)


Optuna Val AUC:   0.7093
Holdout Test AUC: 0.7124


### Whole training data set

In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline


# Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_cat[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_cat[split_index:]

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights   
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
        "class_weight": "balanced", # Adjust weights inversely proportional to class frequencies
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingClassifier(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict_proba(X_val)[:, 1]
        cv_scores.append(roc_auc_score(y_val, preds))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best AUC for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST AUC: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_full_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_full_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Classification/optuna_histgbm_full_parallel.html")

[I 2026-04-23 18:24:58,049] A new study created in memory with name: no-name-f4e10f02-edf5-4bf8-b2cc-71b91b25ea1d
[W 2026-04-23 18:25:01,167] Trial 7 failed with parameters: {'max_iter': 143, 'learning_rate': 0.013715578216161832, 'max_depth': 10, 'min_samples_leaf': 31, 'max_features': 0.7860584571322597, 'max_leaf_nodes': 372, 'l2_regularization': 79.84647563521074, 'max_bins': 420} because of the following error: InvalidParameterError("The 'max_bins' parameter of HistGradientBoostingClassifier must be an int in the range [2, 255]. Got 420 instead.").
Traceback (most recent call last):
  File "/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/pt/4rypzgx161xbyqfj39t1nrqh0000gn/T/ipykernel_82703/3160472742.py", line 59, in objective_histgbm
    full_pipeline.fit(X_tr, y_tr)
  File "/Users/oskarklima/Documents

KeyboardInterrupt: 

[I 2026-04-23 18:28:35,292] Trial 0 finished with value: 0.7348398789598615 and parameters: {'max_iter': 318, 'learning_rate': 0.03619179931386752, 'max_depth': 19, 'min_samples_leaf': 102, 'max_features': 0.895964581643484, 'max_leaf_nodes': 14, 'l2_regularization': 0.0013100115943084365, 'max_bins': 108}. Best is trial 0 with value: 0.7348398789598615.
[I 2026-04-23 18:30:20,176] Trial 2 finished with value: 0.7191617074511787 and parameters: {'max_iter': 607, 'learning_rate': 0.0013737458799797147, 'max_depth': 4, 'min_samples_leaf': 414, 'max_features': 0.6080603549763262, 'max_leaf_nodes': 420, 'l2_regularization': 90.77656462366532, 'max_bins': 215}. Best is trial 0 with value: 0.7348398789598615.
[I 2026-04-23 18:33:44,750] Trial 4 finished with value: 0.7338305409935048 and parameters: {'max_iter': 821, 'learning_rate': 0.00934961791633918, 'max_depth': 5, 'min_samples_leaf': 364, 'max_features': 0.8466727822970896, 'max_leaf_nodes': 212, 'l2_regularization': 0.0013315468221556

In [ ]:
best_params = study_histgbm.best_params.copy()

best_params["max_iter"] = int(best_params["max_iter"] * 10)
best_params["learning_rate"] = best_params["learning_rate"] / 10

best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingClassifier(**best_params)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict_proba(X_val_holdout)[:, 1]
holdout_auc = roc_auc_score(y_val_holdout, holdout_preds)

print("\n" + "="*40)
print(f"Optuna Val AUC:   {study_histgbm.best_value:.4f}")
print(f"Holdout Test AUC: {holdout_auc:.4f}")
print("="*40)

Optuna Val AUC: 0.7566
Holdout Test AUC: 0.7298


## Regression

[Parameters](https://scikit-learn.org/1.6/modules/generated/sklearn.ensemble.HistGradientBoostingRegressor.html)

### 1k

In [ ]:
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline

cols_to_nominal_cat = train_df.select_dtypes(include=["object", "category"]).columns.tolist()

print("Categorical columns:")
for col in cols_to_nominal_cat:
    print(f"- {col}")

cardinality = train_df[cols_to_nominal_cat].nunique()
threshold_for_ohe = 5

cols_for_ohe = cardinality[cardinality <= threshold_for_ohe].index.tolist()
cols_for_te = cardinality[cardinality > threshold_for_ohe].index.tolist()

ohe_categories = []
for col in cols_for_ohe:
    unique_cats = train_df[col].dropna().unique().tolist()
    ohe_categories.append(unique_cats)

ohe_transformer = OneHotEncoder(
    categories=ohe_categories, 
    drop="if_binary", 
    handle_unknown="ignore", 
    sparse_output=False
)

target_transformer_nominal = TargetEncoder(target_type="continuous", smooth="auto")

numeric_preprocessor = ColumnTransformer(
    transformers=[
        ("ohe", ohe_transformer, cols_for_ohe),
        ("target_enc", target_transformer_nominal, cols_for_te)
    ],
    remainder="passthrough", 
    verbose_feature_names_out=False
).set_output(transform="pandas")

imputed_numeric_preprocessor = make_pipeline(
    numeric_preprocessor,
    SimpleImputer(strategy="median")
).set_output(transform="pandas")

gbdt_preprocessor = "passthrough"

Categorical columns identified for encoding:
- home_ownership
- verification_status
- purpose
- addr_state
- initial_list_status
- application_type
- disbursement_method


In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline


# Parameter tuning settings

timeout_seconds = 60*60*0.5
no_improvement_trials = 100

# Split the 1k dataset to evaluate on holdout after tuning
# Had problem with overfitting and extreme bad results on last 200 rows so switched the order

X_tuning = train_1k_X.tail(800)
y_tuning = train_1k_y_reg.tail(800)
X_val_holdout = train_1k_X.head(200)
y_val_holdout = train_1k_y_reg.head(200)

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_1k_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_1k_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_1k_parallel.html")


[I 2026-04-23 21:55:00,629] A new study created in memory with name: no-name-d0ce3147-8dad-47bb-8b4e-010c039cc28b
[I 2026-04-23 21:55:03,876] Trial 0 finished with value: 0.31826604316240986 and parameters: {'max_iter': 168, 'learning_rate': 0.012148700343376246, 'max_depth': 2, 'min_samples_leaf': 83, 'max_features': 0.31229403851151255, 'max_leaf_nodes': 3, 'l2_regularization': 0.4974521928573427, 'max_bins': 229}. Best is trial 0 with value: 0.31826604316240986.
[I 2026-04-23 21:55:04,649] Trial 7 finished with value: 0.3200371953027039 and parameters: {'max_iter': 147, 'learning_rate': 0.005523335307692098, 'max_depth': 6, 'min_samples_leaf': 85, 'max_features': 0.7863599333676317, 'max_leaf_nodes': 23, 'l2_regularization': 0.0425540356280331, 'max_bins': 105}. Best is trial 0 with value: 0.31826604316240986.
[I 2026-04-23 21:55:06,933] Trial 2 finished with value: 0.34228645441836036 and parameters: {'max_iter': 102, 'learning_rate': 0.18348844738134998, 'max_depth': 5, 'min_sampl


BEST RMSE: 0.3155
BEST PARAMETERS:
best_params = {
    "max_iter": 443,
    "learning_rate": 0.00579920597763066,
    "max_depth": 2,
    "min_samples_leaf": 50,
    "max_features": 0.20895930832007648,
    "max_leaf_nodes": 12,
    "l2_regularization": 1.2009666612705499,
    "max_bins": 252,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.4839
  min_samples_leaf    : 0.2045
  max_iter            : 0.0822
  max_bins            : 0.0752
  max_features        : 0.0579
  max_leaf_nodes      : 0.0483
  max_depth           : 0.0433
  l2_regularization   : 0.0046


In [ ]:
# Check overfit on holdout set with best parameters from tuning

best_params = study_histgbm.best_params.copy()
best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")


final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingRegressor(**best_params)
)

final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print(f"Optuna Val RMSE: {study_histgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")

Optuna Val RMSE: 0.3155
Holdout Test RMSE: 0.3376


### 10k

In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline

# Parameter tuning settings

timeout_seconds = 60*60*1
no_improvement_trials = 100

# Split the 10k dataset to evaluate on holdout after tuning

X_tuning = train_10k_X.head(8000)
y_tuning = train_10k_y_reg.head(8000)
X_val_holdout = train_10k_X.tail(2000)
y_val_holdout = train_10k_y_reg.tail(2000)

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 8), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 100), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 3, 31), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_10k_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_10k_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_10k_parallel.html")

[I 2026-04-23 21:56:31,119] A new study created in memory with name: no-name-dc5e1d3f-d0c0-484b-ae47-3dad3a5cc192
[I 2026-04-23 21:56:43,471] Trial 5 finished with value: 0.2981007197095523 and parameters: {'max_iter': 324, 'learning_rate': 0.018094229408000605, 'max_depth': 2, 'min_samples_leaf': 35, 'max_features': 0.38112321528942766, 'max_leaf_nodes': 18, 'l2_regularization': 0.008837154806842915, 'max_bins': 148}. Best is trial 5 with value: 0.2981007197095523.
[I 2026-04-23 21:56:51,993] Trial 4 finished with value: 0.303005021366328 and parameters: {'max_iter': 502, 'learning_rate': 0.06478957931711249, 'max_depth': 2, 'min_samples_leaf': 15, 'max_features': 0.5547570546709562, 'max_leaf_nodes': 13, 'l2_regularization': 0.0010934466249352445, 'max_bins': 70}. Best is trial 5 with value: 0.2981007197095523.
[I 2026-04-23 21:56:59,437] Trial 1 finished with value: 0.2986973313645337 and parameters: {'max_iter': 689, 'learning_rate': 0.022698593237437035, 'max_depth': 2, 'min_sampl


BEST RMSE: 0.2981
BEST PARAMETERS:
best_params = {
    "max_iter": 324,
    "learning_rate": 0.018094229408000605,
    "max_depth": 2,
    "min_samples_leaf": 35,
    "max_features": 0.38112321528942766,
    "max_leaf_nodes": 18,
    "l2_regularization": 0.008837154806842915,
    "max_bins": 148,
}

--- PARAMETER IMPORTANCE ---
  learning_rate       : 0.6706
  max_bins            : 0.1336
  max_leaf_nodes      : 0.0477
  l2_regularization   : 0.0412
  max_features        : 0.0376
  max_iter            : 0.0348
  min_samples_leaf    : 0.0187
  max_depth           : 0.0159


In [ ]:
best_params = study_histgbm.best_params.copy()

best_params["max_iter"] = int(best_params["max_iter"] * 2)
best_params["learning_rate"] = best_params["learning_rate"] / 2

best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingRegressor(**best_params)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_histgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)


Optuna Val RMSE:   0.2981
Holdout Test RMSE: 0.3135


### 100k

In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline


# Parameter tuning settings

timeout_seconds = 60*60*3
no_improvement_trials = 100

# Split the 100k dataset to evaluate on holdout after tuning

X_tuning = train_100k_X.head(80000)
y_tuning = train_100k_y_reg.head(80000)
X_val_holdout = train_100k_X.tail(20000)
y_val_holdout = train_100k_y_reg.tail(20000)

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=5)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_100k_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_100k_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_100k_parallel.html")

[I 2026-04-23 21:57:57,026] A new study created in memory with name: no-name-2c1cf784-9844-495f-bbf5-d6c7401ac779
[I 2026-04-23 21:58:10,514] Trial 2 finished with value: 0.30028758288314183 and parameters: {'max_iter': 962, 'learning_rate': 0.13845820115166996, 'max_depth': 3, 'min_samples_leaf': 272, 'max_features': 0.66106192983912, 'max_leaf_nodes': 179, 'l2_regularization': 6.177836593173464e-05, 'max_bins': 82}. Best is trial 2 with value: 0.30028758288314183.
[I 2026-04-23 21:58:21,360] Trial 8 finished with value: 0.3037145688911723 and parameters: {'max_iter': 122, 'learning_rate': 0.013648304344179138, 'max_depth': 2, 'min_samples_leaf': 128, 'max_features': 0.6612277298031721, 'max_leaf_nodes': 109, 'l2_regularization': 3.902611576236821, 'max_bins': 160}. Best is trial 2 with value: 0.30028758288314183.
[I 2026-04-23 21:58:51,750] Trial 3 finished with value: 0.30173073017017493 and parameters: {'max_iter': 664, 'learning_rate': 0.006816386848284008, 'max_depth': 2, 'min_sa


BEST RMSE: 0.3001
BEST PARAMETERS:
best_params = {
    "max_iter": 786,
    "learning_rate": 0.03553844822139582,
    "max_depth": 14,
    "min_samples_leaf": 461,
    "max_features": 0.6790819451664912,
    "max_leaf_nodes": 187,
    "l2_regularization": 0.015246227631222931,
    "max_bins": 178,
}

--- PARAMETER IMPORTANCE ---
  min_samples_leaf    : 0.8138
  learning_rate       : 0.0497
  max_iter            : 0.0406
  max_features        : 0.0363
  max_bins            : 0.0333
  max_leaf_nodes      : 0.0216
  max_depth           : 0.0042
  l2_regularization   : 0.0004


In [ ]:
best_params = study_histgbm.best_params.copy()

best_params["max_iter"] = int(best_params["max_iter"] * 10)
best_params["learning_rate"] = best_params["learning_rate"] / 10

best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingRegressor(**best_params)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_histgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)


Optuna Val RMSE:   0.3001
Holdout Test RMSE: 0.2989


### Whole training data set

In [ ]:
import optuna
import numpy as np
import warnings
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import TimeSeriesSplit
from optuna.visualization import (
    plot_optimization_history, 
    plot_param_importances, 
    plot_parallel_coordinate
)
from optuna.samplers import TPESampler
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline


# Parameter tuning settings

timeout_seconds = 60*60*4
no_improvement_trials = 100

# Split the dataset to evaluate on holdout after tuning

split_index = int(len(train_full_X) * 0.8)

X_tuning = train_full_X[:split_index]
y_tuning = train_full_y_reg[:split_index]
X_val_holdout = train_full_X[split_index:]
y_val_holdout = train_full_y_reg[split_index:]

def objective_histgbm(trial):
    params = {
        "max_iter": trial.suggest_int("max_iter", 100, 1000), # Number of boosting rounds
        "learning_rate": trial.suggest_float("learning_rate", 0.001, 0.2, log=True), # Step size
        "max_depth": trial.suggest_int("max_depth", 2, 20), # Max tree depth
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 10, 500), # Min samples required in a leaf node
        "max_features": trial.suggest_float("max_features", 0.2, 1.0), # Number of features to consider when looking for the best split
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 5, 512), # Max leaf nodes in the tree
        "l2_regularization": trial.suggest_float("l2_regularization", 1e-5, 100.0, log=True), # L2 regularization term on weights   
        "random_state": 42, # For reproducibility
        "verbose": 0, # No verbose output during training
        "max_bins": trial.suggest_int("max_bins", 63, 255), # Maximum number of bins to use for discretizing continuous features
    }

    full_pipeline = make_pipeline(
        numeric_preprocessor,
        HistGradientBoostingRegressor(**params)
    )
    
    tscv = TimeSeriesSplit(n_splits=3)
    cv_scores = []
    
    for train_idx, val_idx in tscv.split(X_tuning):
        X_tr, X_val = X_tuning.iloc[train_idx], X_tuning.iloc[val_idx]
        y_tr, y_val = y_tuning.iloc[train_idx], y_tuning.iloc[val_idx]
        
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            full_pipeline.fit(X_tr, y_tr)
        
        preds = full_pipeline.predict(X_val)
        cv_scores.append(np.sqrt(mean_squared_error(y_val, preds)))
        
    return np.mean(cv_scores)

def stop_optuna(study, trial): # Custom callback to stop optimization if no improvement in best RMSE for a certain number of trials
    if study.best_trial.number + no_improvement_trials <= trial.number:
        print(f"\n[Early Stop] No improvement in {no_improvement_trials} trials. Stopping optimization.")
        study.stop()

study_histgbm = optuna.create_study(direction="minimize", sampler=TPESampler(seed=42))
study_histgbm.optimize(objective_histgbm, timeout=timeout_seconds, n_jobs=-1, callbacks=[stop_optuna])

print("\n" + "="*40)
print(f"BEST RMSE: {study_histgbm.best_value:.4f}")
print("BEST PARAMETERS:")

print("best_params = {")
for k, v in study_histgbm.best_params.items():
    if isinstance(v, str):
        print(f'    "{k}": "{v}",')
    else:
        print(f'    "{k}": {v},')
print("}")

print("\n--- PARAMETER IMPORTANCE ---")
importances = optuna.importance.get_param_importances(study_histgbm)
for param, importance in importances.items():
    print(f"  {param:20}: {importance:.4f}")
print("="*40)


fig1 = plot_optimization_history(study_histgbm)
fig1.show()
    
fig2 = plot_param_importances(study_histgbm)
fig2.show()
    
fig3 = plot_parallel_coordinate(study_histgbm)
fig3.show()
    
fig1.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_full_history.html")
fig2.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_full_importance.html")
fig3.write_html("hyperparameter_tuning/HistGBM/Regression/optuna_histgbm_full_parallel.html")

[I 2026-04-23 18:24:58,049] A new study created in memory with name: no-name-f4e10f02-edf5-4bf8-b2cc-71b91b25ea1d
[W 2026-04-23 18:25:01,167] Trial 7 failed with parameters: {'max_iter': 143, 'learning_rate': 0.013715578216161832, 'max_depth': 10, 'min_samples_leaf': 31, 'max_features': 0.7860584571322597, 'max_leaf_nodes': 372, 'l2_regularization': 79.84647563521074, 'max_bins': 420} because of the following error: InvalidParameterError("The 'max_bins' parameter of HistGradientBoostingClassifier must be an int in the range [2, 255]. Got 420 instead.").
Traceback (most recent call last):
  File "/Users/oskarklima/Documents/Oskar/Škola/Bakalářka/Praktická část/Github/bachelor_thesis/.venv/lib/python3.10/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
  File "/var/folders/pt/4rypzgx161xbyqfj39t1nrqh0000gn/T/ipykernel_82703/3160472742.py", line 59, in objective_histgbm
    full_pipeline.fit(X_tr, y_tr)
  File "/Users/oskarklima/Documents

KeyboardInterrupt: 

[I 2026-04-23 18:28:35,292] Trial 0 finished with value: 0.7348398789598615 and parameters: {'max_iter': 318, 'learning_rate': 0.03619179931386752, 'max_depth': 19, 'min_samples_leaf': 102, 'max_features': 0.895964581643484, 'max_leaf_nodes': 14, 'l2_regularization': 0.0013100115943084365, 'max_bins': 108}. Best is trial 0 with value: 0.7348398789598615.
[I 2026-04-23 18:30:20,176] Trial 2 finished with value: 0.7191617074511787 and parameters: {'max_iter': 607, 'learning_rate': 0.0013737458799797147, 'max_depth': 4, 'min_samples_leaf': 414, 'max_features': 0.6080603549763262, 'max_leaf_nodes': 420, 'l2_regularization': 90.77656462366532, 'max_bins': 215}. Best is trial 0 with value: 0.7348398789598615.
[I 2026-04-23 18:33:44,750] Trial 4 finished with value: 0.7338305409935048 and parameters: {'max_iter': 821, 'learning_rate': 0.00934961791633918, 'max_depth': 5, 'min_samples_leaf': 364, 'max_features': 0.8466727822970896, 'max_leaf_nodes': 212, 'l2_regularization': 0.0013315468221556

In [ ]:
best_params = study_histgbm.best_params.copy()

best_params["max_iter"] = int(best_params["max_iter"] * 10)
best_params["learning_rate"] = best_params["learning_rate"] / 10

best_params["random_state"] = 42
best_params["verbose"] = 0
print(f"BEST PARAMS: {best_params}")

final_pipeline = make_pipeline(
    numeric_preprocessor,
    HistGradientBoostingRegressor(**best_params)
)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    final_pipeline.fit(X_tuning, y_tuning)

holdout_preds = final_pipeline.predict(X_val_holdout)
holdout_rmse = np.sqrt(mean_squared_error(y_val_holdout, holdout_preds))

print("\n" + "="*40)
print(f"Optuna Val RMSE:   {study_histgbm.best_value:.4f}")
print(f"Holdout Test RMSE: {holdout_rmse:.4f}")
print("="*40)

Optuna Val AUC: 0.7566
Holdout Test AUC: 0.7298
